Run nnU-Net's pretrained KiTS2021 model (kidney+tumor+cyst) on KiTS19 CT
volumes inside a Kaggle notebook — no training, just inference.

Upload this notebook to Kaggle as-is and run all cells top to bottom.
Turn on internet access first (Settings > Internet > On) — this pulls
segmentation labels from GitHub and CT volumes from Hugging Face, no Kaggle
Dataset attachment needed.

[Cell 1] Install nnU-Net v1 — the version that ships pretrained models
(nnunetv2 has none, confirmed against its docs, 2026-08)

In [ ]:
!pip install -q nnunet

[Cell 2] Paths, dataset id, and case budget

In [ ]:
# /kaggle/input is read-only, so scratch space for downloaded/organized data
# must live under /kaggle/working (the only writable, persisted-as-output location).
import os
from pathlib import Path

WORK = Path("/kaggle/working")
KITS19_REPO = WORK / "kits19"  # git clone target; labels live under kits19/data/case_*/segmentation.nii.gz
DATASET_ID = "137"
DATASET_NAME = f"Dataset{DATASET_ID}_KiTS19"

os.environ["nnUNet_raw"] = str(WORK / "nnUNet_raw")

Path(os.environ["nnUNet_raw"]).mkdir(parents=True, exist_ok=True)

# Each CT volume is a multi-hundred-MB download and gets copied a second time in
# Cell 4, while the model alone eats 3.5 GB of /kaggle/working's ~20 GB. Start
# small to prove the pipeline runs end to end.
MAX_CASES = 5  # ponytail: raise (up to 300) once a small run succeeds; watch disk

[Cell 3] Fetch KiTS19: labels via git clone, imaging volumes via Hugging Face

In [ ]:
# The official repo's own download script (starter_code/get_imaging.py) always
# fetches all 300 cases with no subset option, so this reimplements just enough
# of it to respect MAX_CASES. Source verified against neheller/kits19 (2026-08).
import requests
from tqdm import tqdm

if not KITS19_REPO.exists():
    os.system(f"git clone --depth 1 https://github.com/neheller/kits19.git {KITS19_REPO}")

HF_BASE_URL = "https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/main"

for i in range(MAX_CASES):
    case_id = f"case_{i:05d}"
    case_dir = KITS19_REPO / "data" / case_id
    imaging_path = case_dir / "imaging.nii.gz"
    if not case_dir.exists() or not (case_dir / "segmentation.nii.gz").exists():
        print(f"Skipping {case_id}: no segmentation label in the cloned repo.")
        continue
    if imaging_path.exists():
        continue

    response = requests.get(f"{HF_BASE_URL}/images/{case_id}.nii.gz", stream=True)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    tmp_path = imaging_path.with_suffix(".tmp")
    with tmp_path.open("wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=case_id) as bar:
        for chunk in response.iter_content(chunk_size=1 << 20):
            f.write(chunk)
            bar.update(len(chunk))
    tmp_path.rename(imaging_path)

[Cell 4] Link cases into the flat `<case>_0000.nii.gz` layout nnU-Net expects

In [ ]:
import shutil

raw_dataset_dir = Path(os.environ["nnUNet_raw"]) / DATASET_NAME
images_dir = raw_dataset_dir / "imagesTr"
images_dir.mkdir(parents=True, exist_ok=True)

case_dirs = sorted(
    p for p in (KITS19_REPO / "data").glob("case_*")
    if (p / "imaging.nii.gz").exists()
)[:MAX_CASES]
if not case_dirs:
    raise FileNotFoundError("No downloaded case has imaging.nii.gz — check Cell 3's output.")

# A previous run with a larger MAX_CASES leaves its volumes on disk, and
# /kaggle/working only gets ~20 GB. Drop the ones this run will not use.
keep = {p.name for p in case_dirs}
for stale in sorted((KITS19_REPO / "data").glob("case_*/imaging.nii.gz")):
    if stale.parent.name not in keep:
        stale.unlink()

# Inference never reads labels, so a labelsTr from an earlier run is dead weight.
shutil.rmtree(raw_dataset_dir / "labelsTr", ignore_errors=True)

# Hard-link instead of copy. nnU-Net only needs a flat dir of <case>_0000.nii.gz,
# and a second physical copy of every CT volume is exactly what filled the disk.
# Source and dest are both under /kaggle/working (same filesystem), so os.link is
# free and instant. Relink unconditionally: a partial file from a run that died
# mid-copy would otherwise be kept and silently fed to the model.
for case_dir in case_dirs:
    case_id = case_dir.name  # e.g. "case_00000"
    dest = images_dir / f"{case_id}_0000.nii.gz"
    dest.unlink(missing_ok=True)
    os.link(case_dir / "imaging.nii.gz", dest)

usage = shutil.disk_usage(WORK)
print(f"Linked {len(case_dirs)} cases into {images_dir}")
print(f"Disk: {usage.used / 2**30:.1f} GiB used, {usage.free / 2**30:.1f} GiB free")

[Cell 5] Run the pretrained model

In [ ]:
import shutil
import subprocess
import time
import zipfile

# nnU-Net v1 uses its own env var names — without these it exits silently.
V1_WORK = WORK / "nnunet_v1"
os.environ["nnUNet_raw_data_base"] = str(V1_WORK / "raw_data_base")
os.environ["nnUNet_preprocessed"] = str(V1_WORK / "preprocessed")
os.environ["RESULTS_FOLDER"] = str(V1_WORK / "results")
for path in os.environ["nnUNet_raw_data_base"], os.environ["nnUNet_preprocessed"], os.environ["RESULTS_FOLDER"]:
    Path(path).mkdir(parents=True, exist_ok=True)

# nnunet 1.7.1 (the PyPI release) calls torch.load() without weights_only=, and
# PyTorch >= 2.6 defaults that to True, which refuses these pickled checkpoints.
# torch.serialization honours this env var only when the callsite omitted the
# arg — exactly this case — and unlike a monkeypatch it survives into the
# nnUNet_predict subprocess. Safe here: weights come from DKFZ's own Zenodo record.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"  # keep the log current rather than block-buffered

predictions_dir = WORK / "predictions_pretrained"
predictions_dir.mkdir(exist_ok=True)

# `nnUNet_download_pretrained_model` is broken as of 2026-08 — it hardcodes an
# HTTP/1.0 downgrade that Zenodo now 404s on (MIC-DKFZ/nnUNet#2876), so this
# downloads + extracts the zip directly.
# The zip's top level is `3d_fullres/`, and nnU-Net resolves models under
# $RESULTS_FOLDER/nnUNet (nnunet/paths.py: network_training_output_dir), so the
# extract MUST target that subdir — extracting to $RESULTS_FOLDER instead hides
# the model and nnUNet_predict dies with "Could not find a task with the ID 135".
# Swap the URL + id for another v1 task (same <case>_0000.nii.gz convention):
# Task003_Liver, Task007_Pancreas, Task008_HepaticVessel, Task010_Colon — all
# at zenodo.org/record/4003545.
PRETRAINED_MODEL_URL = "https://zenodo.org/record/5126443/files/Task135_KiTS2021.zip?download=1"
PRETRAINED_TASK_ID = "135"
CONFIGURATION = "3d_fullres"

results_root = Path(os.environ["RESULTS_FOLDER"])
model_root = results_root / "nnUNet" / CONFIGURATION

# An earlier version of this cell extracted one level too high. That stale tree
# is ~3.5 GB of pure waste and only ever exists because of that bug, so clear it.
for stale in ("2d", "3d_lowres", "3d_fullres", "3d_cascade_fullres"):
    shutil.rmtree(results_root / stale, ignore_errors=True)

zip_path = WORK / "pretrained_model.zip"
if not list(model_root.glob(f"Task{PRETRAINED_TASK_ID}*")):
    if not zip_path.exists():
        response = requests.get(PRETRAINED_MODEL_URL, stream=True, timeout=100)
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with zip_path.open("wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="pretrained model") as bar:
            for chunk in response.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(results_root / "nnUNet")
zip_path.unlink(missing_ok=True)  # 3.5 GB, and /kaggle/working only gets ~20 GB

# Fail here with the actual path rather than inside nnUNet_predict's generic exit 1.
task_dirs = sorted(model_root.glob(f"Task{PRETRAINED_TASK_ID}*"))
if not task_dirs:
    raise FileNotFoundError(f"No Task{PRETRAINED_TASK_ID}* under {model_root} — extraction landed elsewhere.")
usage = shutil.disk_usage(WORK)
print(f"Model ready: {task_dirs[0]}")
print(f"Disk: {usage.used / 2**30:.1f} GiB used, {usage.free / 2**30:.1f} GiB free")

import torch  # noqa: E402 - only used for the warning below
if not torch.cuda.is_available():
    print("WARNING: no GPU visible. 3d_fullres on CPU is hours per case — "
          "set Settings > Accelerator to a GPU before running this.")

cmd = [
    "nnUNet_predict",
    "-i", str(images_dir),
    "-o", str(predictions_dir),
    "-t", PRETRAINED_TASK_ID, "-m", CONFIGURATION,
    "-f", "0",  # single fold instead of the full 5-fold ensemble; drop for full ensemble accuracy
    # Test-time augmentation (8x mirroring) is ON by default; nnU-Net's own help
    # puts the cost at "roughly factor 8" in 3D for a small accuracy gain.
    "--disable_tta",
    # Default is 6 background workers, each holding a full CT volume in RAM.
    "--num_threads_preprocessing", "2",
]

# nnU-Net reports no progress of its own, but it writes one .nii.gz per finished
# case — so counting those drives a real bar with an ETA. Its own output goes to
# a log rather than the notebook, otherwise it would trample the bar; the tail is
# printed if it fails. nnU-Net skips cases already present in the output folder,
# so a re-run resumes instead of starting over — hence `initial=`.
n_cases = len(list(images_dir.glob("*_0000.nii.gz")))
already_done = len(list(predictions_dir.glob("case_*.nii.gz")))
log_path = WORK / "nnunet_predict.log"

with log_path.open("w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    with tqdm(total=n_cases, initial=already_done, unit="case", desc="predicting") as bar:
        while proc.poll() is None:
            time.sleep(5)
            bar.update(len(list(predictions_dir.glob("case_*.nii.gz"))) - bar.n)
        bar.update(len(list(predictions_dir.glob("case_*.nii.gz"))) - bar.n)

if proc.returncode != 0:
    print(log_path.read_text()[-4000:])
    raise RuntimeError(f"nnUNet_predict exited with {proc.returncode} — full log at {log_path}")
print(f"Pretrained predictions saved to {predictions_dir}")

[Cell 6] Dice against the KiTS19 ground truth

In [ ]:
# nnU-Net prints nothing about quality, so score the predictions against KiTS19's
# own labels. KiTS2021 predicts 0 bg / 1 kidney / 2 tumor / 3 cyst, but KiTS19 has
# only kidney and tumor and those cyst voxels sit inside its kidney label — so
# predicted cysts are merged into kidney first, or kidney Dice is unfairly low.
import numpy as np
import nibabel as nib


def dice(pred, truth, label):
    p, t = pred == label, truth == label
    denom = p.sum() + t.sum()
    # Neither predicted nor present: undefined, not zero. nanmean drops it below.
    return np.nan if denom == 0 else 2.0 * np.logical_and(p, t).sum() / denom


rows = []
for pred_path in sorted(predictions_dir.glob("case_*.nii.gz")):
    case_id = pred_path.name.removesuffix(".nii.gz")
    truth_path = KITS19_REPO / "data" / case_id / "segmentation.nii.gz"
    if not truth_path.exists():
        print(f"{case_id}: no ground truth, skipped")
        continue
    pred = np.asanyarray(nib.load(pred_path).dataobj)
    truth = np.asanyarray(nib.load(truth_path).dataobj)
    if pred.shape != truth.shape:
        print(f"{case_id}: shape {pred.shape} vs {truth.shape}, skipped")
        continue
    pred = np.where(pred == 3, 1, pred)  # cyst -> kidney, to match KiTS19
    rows.append((case_id, dice(pred, truth, 1), dice(pred, truth, 2)))

for case_id, kidney, tumor in rows:
    print(f"{case_id}  kidney {kidney:.3f}  tumor {tumor:.3f}")

if rows:
    scores = np.array([[k, t] for _, k, t in rows], float)
    print(f"\nmean over {len(rows)} cases  "
          f"kidney {np.nanmean(scores[:, 0]):.3f}  tumor {np.nanmean(scores[:, 1]):.3f}")
else:
    print("No predictions scored — did Cell 5 finish?")